In [ ]:
# %% [markdown]
# # Quantum Chemical Reaction Studio: $Cl^\bullet + H_2 \rightarrow HCl + H^\bullet$
# Adjust the atomic distances to explore the reaction pathway and locate the transition state.

# %%
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML
import py3Dmol

# ==========================================
# 1. Physics Engine: LEPS Quantum Potential
# ==========================================
def morse_components(r, De, re, beta, S):
    exp_t = np.exp(-beta * (r - re))
    denom = 4.0 * (1.0 + S)
    Q = (De / denom) * ((3.0 + S) * (exp_t**2) - (2.0 + 6.0 * S) * exp_t)
    J = (De / denom) * ((1.0 + 3.0 * S) * (exp_t**2) - (6.0 + 2.0 * S) * exp_t)
    return Q, J

def calculate_energy(r1, r2):
    """
    Computes relative potential energy (kJ/mol) for linear Cl-H(1)-H(2).
    r1: distance Cl-H(1) in Angstroms
    r2: distance H(1)-H(2) in Angstroms
    Reference: Cl + H2 (reactants at infinite separation) = 0.0 kJ/mol
    """
    r3 = r1 + r2  # Collinear geometry
    # Parameters: De (kJ/mol), re (Å), beta (Å^-1), Sato parameter S
    De_clh, re_clh, beta_clh, S_clh = 445.0, 1.275, 1.87, 0.15
    De_hh,  re_hh,  beta_hh,  S_hh  = 458.0, 0.741, 1.94, 0.15

    Q1, J1 = morse_components(r1, De_clh, re_clh, beta_clh, S_clh)
    Q2, J2 = morse_components(r2, De_hh,  re_hh,  beta_hh,  S_hh)
    Q3, J3 = morse_components(r3, De_clh, re_clh, beta_clh, S_clh)

    E_abs = Q1 + Q2 + Q3 - np.sqrt(0.5 * ((J1 - J2)**2 + (J2 - J3)**2 + (J3 - J1)**2))
    # Relative to isolated Cl + H2 at equilibrium:
    E_rel = E_abs + De_hh
    return float(E_rel)

# Precompute the PES grid for plotting
r1_grid = np.linspace(1.0, 3.2, 55)
r2_grid = np.linspace(0.6, 2.6, 55)
R1, R2 = np.meshgrid(r1_grid, r2_grid)
Z_pes = np.vectorize(calculate_energy)(R1, R2)
Z_pes_clipped = np.clip(Z_pes, -50, 200)

# ==========================================
# 2. Molecular Viewer Helper
# ==========================================
def make_molecule_html(r1, r2):
    xyz = f"3\nCl-H-H Collinear\nCl 0.0 0.0 0.0\nH 0.0 0.0 {r1:.3f}\nH 0.0 0.0 {r1+r2:.3f}"
    viewer = py3Dmol.view(width=340, height=220)
    viewer.addModel(xyz, "xyz")
    viewer.setStyle({'elem': 'Cl'}, {'sphere': {'radius': 0.65, 'color': '#2ca02c'}})
    viewer.setStyle({'elem': 'H'}, {'sphere': {'radius': 0.35, 'color': '#ffffff'}})
    # Add cylinder bonds if atoms are close enough
    if r1 <= 1.8:
        viewer.addCylinder({'start': {'x': 0, 'y': 0, 'z': 0}, 'end': {'x': 0, 'y': 0, 'z': r1}, 'radius': 0.1, 'color': '#888888'})
    if r2 <= 1.3:
        viewer.addCylinder({'start': {'x': 0, 'y': 0, 'z': r1}, 'end': {'x': 0, 'y': 0, 'z': r1 + r2}, 'radius': 0.1, 'color': '#888888'})
    viewer.zoomTo()
    return viewer._make_html()

# ==========================================
# 3. Dynamic ORCA Output Log Generator
# ==========================================
def generate_orca_log(r1, r2, e_val):
    hartree = -460.124589 + (e_val / 2625.5)
    return f"""--------------------------------------------------
                  ORCA 5.0.3
  Quantum Chemistry Program Package (MPI-Parallel)
--------------------------------------------------
Method: B3LYP/def2-TZVP (UKS, Open-Shell)
Coordinates (Angstrom):
  Cl   0.000000   0.000000   0.000000
  H    0.000000   0.000000   {r1:.4f}
  H    0.000000   0.000000   {r1+r2:.4f}

SCF CONVERGENCE:
  Cycle     Energy (Eh)         dE (Eh)       Max Gradiant
    01   {hartree - 0.00412:.6f}   -0.0000000   0.041235
    05   {hartree - 0.00003:.6f}   -0.0000104   0.001201
    08   {hartree:.6f}   -0.0000001   0.000012  [CONVERGED]

Total Energy         :  {hartree:.8f} Eh
Relative Energy (dE) :  {e_val:+.2f} kJ/mol (vs Reactants)
<S**2> Expectation   :  0.7512 (Doublet radical state)
Status               :  NORMAL TERMINATION
"""

# ==========================================
# 4. State Management and Visual Widgets
# ==========================================
history_records = []

slider_r1 = widgets.FloatSlider(
    value=2.40, min=1.00, max=3.20, step=0.05,
    description='r(Cl-H) [Å]:', readout_format='.2f',
    layout=widgets.Layout(width='90%')
)
slider_r2 = widgets.FloatSlider(
    value=0.74, min=0.60, max=2.50, step=0.05,
    description='r(H-H) [Å]:', readout_format='.2f',
    layout=widgets.Layout(width='90%')
)

btn_calc = widgets.Button(description="Calculate Energy", button_style='success', icon='calculator')
btn_reset = widgets.Button(description="Clear History", button_style='warning', icon='trash')

mol_html_widget = widgets.HTML(value=make_molecule_html(slider_r1.value, slider_r2.value))
table_output = widgets.HTML(value="<i>No calculations recorded yet.</i>")
orca_log_box = widgets.Textarea(
    value="Click 'Calculate Energy' to execute the quantum chemical calculation.",
    layout=widgets.Layout(width='100%', height='180px'),
    disabled=True
)

# Plotly 2D Contour Figure
pes_fig = go.FigureWidget(data=[
    go.Contour(
        z=Z_pes_clipped, x=r1_grid, y=r2_grid,
        colorscale='Viridis',
        contours=dict(start=-40, end=180, size=10, showlabels=True),
        colorbar=dict(title='ΔE (kJ/mol)'),
        hoverinfo='x+y+z'
    ),
    go.Scatter(
        x=[], y=[], mode='markers',
        marker=dict(size=10, color='red', symbol='circle', line=dict(width=1, color='white')),
        name='History'
    ),
    go.Scatter(
        x=[slider_r1.value], y=[slider_r2.value], mode='markers',
        marker=dict(size=14, color='orange', symbol='star', line=dict(width=2, color='black')),
        name='Current'
    )
])

pes_fig.update_layout(
    title="2D Potential Energy Surface (Cl ··· H ··· H)",
    xaxis_title="r(Cl-H) Distance [Å]",
    yaxis_title="r(H-H) Distance [Å]",
    width=520, height=420,
    margin=dict(l=50, r=30, t=50, b=50),
    legend=dict(orientation="h", y=-0.2)
)

# ==========================================
# 5. Widget Event Handlers
# ==========================================
def update_geometry_preview(change):
    mol_html_widget.value = make_molecule_html(slider_r1.value, slider_r2.value)
    # Update current position indicator on the contour map
    with pes_fig.batch_update():
        pes_fig.data[2].x = [slider_r1.value]
        pes_fig.data[2].y = [slider_r2.value]

slider_r1.observe(update_geometry_preview, names='value')
slider_r2.observe(update_geometry_preview, names='value')

def on_calculate_clicked(b):
    r1 = round(slider_r1.value, 2)
    r2 = round(slider_r2.value, 2)
    energy = calculate_energy(r1, r2)
    
    # Classify state
    if r1 >= 2.0 and r2 <= 0.85:
        chem_state = "Reactants (Cl· + H₂)"
    elif r1 <= 1.45 and r2 >= 1.50:
        chem_state = "Products (HCl + H·)"
    elif 1.30 <= r1 <= 1.70 and 0.85 <= r2 <= 1.25:
        chem_state = "⚠️ Transition State Zone"
    else:
        chem_state = "Intermediate"

    # Save to history
    history_records.append({
        'Run': len(history_records) + 1,
        'r(Cl-H) [Å]': f"{r1:.2f}",
        'r(H-H) [Å]': f"{r2:.2f}",
        'ΔE [kJ/mol]': f"{energy:+.1f}",
        'Region': chem_state
    })
    
    # Update table
    df = pd.DataFrame(history_records).tail(6)
    table_output.value = df.to_html(index=False, classes='table table-striped table-bordered')
    
    # Update ORCA log box
    orca_log_box.value = generate_orca_log(r1, r2, energy)
    
    # Update Plotly scatter trace
    x_pts = [float(rec['r(Cl-H) [Å]']) for rec in history_records]
    y_pts = [float(rec['r(H-H) [Å]']) for rec in history_records]
    with pes_fig.batch_update():
        pes_fig.data[1].x = x_pts
        pes_fig.data[1].y = y_pts

def on_reset_clicked(b):
    history_records.clear()
    table_output.value = "<i>History cleared.</i>"
    orca_log_box.value = "History cleared."
    with pes_fig.batch_update():
        pes_fig.data[1].x = []
        pes_fig.data[1].y = []

btn_calc.on_click(on_calculate_clicked)
btn_reset.on_click(on_reset_clicked)

# ==========================================
# 6. Dashboard Layout
# ==========================================
controls_box = widgets.VBox([
    widgets.HTML("<h4>1. Adjust Bond Distances</h4>"),
    slider_r1,
    slider_r2,
    widgets.HBox([btn_calc, btn_reset]),
    widgets.HTML("<hr style='margin:10px 0;'><h4>2. Geometry Preview</h4>"),
    mol_html_widget
], layout=widgets.Layout(width='360px', padding='10px'))

plot_and_table_box = widgets.VBox([
    pes_fig,
    widgets.HTML("<h4>Calculation History</h4>"),
    table_output
], layout=widgets.Layout(width='540px', padding='10px'))

accordion = widgets.Accordion(children=[orca_log_box])
accordion.set_title(0, "Raw Quantum Calculation Output (ORCA Log)")

dashboard = widgets.VBox([
    widgets.HTML("""
        <div style="background-color:#20293a; color:white; padding:12px; border-radius:6px; margin-bottom:15px;">
            <h2 style="margin:0;">Chlorine Radical Substitution Experiment</h2>
            <p style="margin:4px 0 0 0; font-size:14px; opacity:0.85;">
                Investigate the barrier separating <b>Cl· + H₂</b> from <b>HCl + H·</b>.
            </p>
        </div>
    """),
    widgets.HBox([controls_box, plot_and_table_box]),
    accordion
])

display(dashboard)